In [1]:
!pip install requests beautifulsoup4 pandas

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import csv
import time

In [3]:
URL = "https://www.amazon.in/gp/aw/d/B0CX5KRJWS"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-IN,en;q=0.9"
}

CSV_FILE = "AmazonWebScraperDataset.csv"

TARGET_PRICE = 299

In [4]:
def scrape_product(url):
    
    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=15
        )
        
        response.raise_for_status()
        
    except requests.RequestException as e:
        print("Error while accessing Amazon:", e)
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    # Product Title
    title_element = soup.find(id="productTitle")
    
    if title_element:
        title = title_element.get_text(strip=True)
    else:
        title = None

    # Product Price
    price_element = soup.select_one("span.a-price-whole")
    
    if price_element:
        price_text = price_element.get_text(strip=True)
        price_text = price_text.replace(",", "")
        
        try:
            price = float(price_text)
        except ValueError:
            price = None
            
    else:
        price = None

    # Rating
    rating_element = soup.select_one(
        "span.a-icon-alt"
    )
    
    if rating_element:
        rating_text = rating_element.get_text(strip=True)
        
        try:
            rating = float(rating_text.split()[0])
        except (ValueError, IndexError):
            rating = None
    else:
        rating = None

    # Number of Reviews
    review_element = soup.select_one(
        "#acrCustomerReviewText"
    )
    
    if review_element:
        review_text = review_element.get_text(strip=True)
        review_text = review_text.replace(",", "")
        
        try:
            reviews = int(review_text.split()[0])
        except (ValueError, IndexError):
            reviews = None
    else:
        reviews = None

    # Availability
    availability_element = soup.select_one(
        "#availability span"
    )
    
    if availability_element:
        availability = availability_element.get_text(
            strip=True
        )
    else:
        availability = "Unknown"

    # Scraping Date & Time
    scraped_at = datetime.now()

    # Create result
    product_data = {
        "title": title,
        "price": price,
        "rating": rating,
        "reviews": reviews,
        "availability": availability,
        "url": url,
        "scraped_at": scraped_at
    }

    return product_data

In [5]:
#test the scraper function
product = scrape_product(URL)

product

{'title': 'T-Shirt for Men – 100% Cotton Mountain T-Shirt Half Sleeve Round Neck Graphic Print | Unisex Regular & Oversized Fit | 180/220 GSM | S–5XL | Black, Blue, Maroon, White, Yellow, Grey',
 'price': 499.0,
 'rating': 4.7,
 'reviews': None,
 'availability': 'In stock',
 'url': 'https://www.amazon.in/gp/aw/d/B0CX5KRJWS',
 'scraped_at': datetime.datetime(2026, 9, 11, 19, 17, 37, 821413)}

In [6]:
#dispalaying the data properly

if product:
    
    print("PRODUCT INFORMATION")
    print("-" * 40)
    
    print("Title:", product["title"])
    print("Price:", product["price"])
    print("Rating:", product["rating"])
    print("Reviews:", product["reviews"])
    print("Availability:", product["availability"])
    print("Scraped At:", product["scraped_at"])

PRODUCT INFORMATION
----------------------------------------
Title: T-Shirt for Men – 100% Cotton Mountain T-Shirt Half Sleeve Round Neck Graphic Print | Unisex Regular & Oversized Fit | 180/220 GSM | S–5XL | Black, Blue, Maroon, White, Yellow, Grey
Price: 499.0
Rating: 4.7
Reviews: None
Availability: In stock
Scraped At: 2026-09-11 19:17:37.821413


In [7]:
def save_to_csv(product, filename=CSV_FILE):
    
    if product is None:
        print("No product data to save.")
        return

    data = pd.DataFrame([product])

    try:
        existing_data = pd.read_csv(filename)
        
        updated_data = pd.concat(
            [existing_data, data],
            ignore_index=True
        )
        
    except FileNotFoundError:
        updated_data = data

    updated_data.to_csv(
        filename,
        index=False
    )

    print("Data saved successfully.")

In [8]:
#save the data

save_to_csv(product)

Data saved successfully.


In [9]:
#reading the data

df = pd.read_csv(CSV_FILE)

df

,title,price,rating,reviews,availability,url,scraped_at
0,T-Shirt for Men – 100% Cotton Mountain T-Shirt...,499.0,4.7,NaN,In stock,https://www.amazon.in/gp/aw/d/B0CX5KRJWS,2026-09-11 11:06:02.588074
1,T-Shirt for Men – 100% Cotton Mountain T-Shirt...,499.0,4.7,NaN,In stock,https://www.amazon.in/gp/aw/d/B0CX5KRJWS,2026-09-11 19:17:37.821413


In [10]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         2 non-null      str    
 1   price         2 non-null      float64
 2   rating        2 non-null      float64
 3   reviews       0 non-null      float64
 4   availability  2 non-null      str    
 5   url           2 non-null      str    
 6   scraped_at    2 non-null      str    
dtypes: float64(3), str(4)
memory usage: 762.0 bytes
None


In [13]:
print(df.describe())

       price  rating  reviews
count    2.0     2.0      0.0
mean   499.0     4.7      NaN
std      0.0     0.0      NaN
min    499.0     4.7      NaN
25%    499.0     4.7      NaN
50%    499.0     4.7      NaN
75%    499.0     4.7      NaN
max    499.0     4.7      NaN


In [15]:
product = scrape_product(URL)

if product is not None and product["price"] is not None and product["price"] < 299:
    send_mail()

In [16]:
def check_price(product, target_price=TARGET_PRICE):
    
    if product is None:
        print("Unable to check price.")
        return False

    price = product["price"]

    print("Product:", product["title"])
    print("Current Price:", price)
    print("Target Price:", target_price)

    if price is None:
        print("Price unavailable.")
        return False

    if price <= target_price:
        print("🎉 Target price reached!")
        return True
    
    else:
        print("Target price not reached.")
        return False

In [17]:
check_price(product)

Product: T-Shirt for Men – 100% Cotton Mountain T-Shirt Half Sleeve Round Neck Graphic Print | Unisex Regular & Oversized Fit | 180/220 GSM | S–5XL | Black, Blue, Maroon, White, Yellow, Grey
Current Price: 499.0
Target Price: 299
Target price not reached.


False

In [18]:
product = scrape_product(URL)

In [19]:
save_to_csv(product)

Data saved successfully.


In [20]:
check_price(product)

Product: T-Shirt for Men – 100% Cotton Mountain T-Shirt Half Sleeve Round Neck Graphic Print | Unisex Regular & Oversized Fit | 180/220 GSM | S–5XL | Black, Blue, Maroon, White, Yellow, Grey
Current Price: 499.0
Target Price: 299
Target price not reached.


False

In [21]:
PRODUCT_URLS = [
"https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd_plhdr=t&hsa_cr_id=0&qid=1789134778&sr=1-1-941ebe59-d25b-49d0-b214-12cc5b66c90f&i=aps&aref=wsdpJlJGBJ&_encoding=UTF8&ref_=sbx_s_sparkle_sbtcd_asin_0_title&pd_rd_w=hkVwn&content-id=amzn1.sym.7b6188a8-103f-4f88-b43a-19aba06e30c1%3Aamzn1.sym.7b6188a8-103f-4f88-b43a-19aba06e30c1&pf_rd_p=7b6188a8-103f-4f88-b43a-19aba06e30c1&pf_rd_r=NSM1EVTP5VX9ACY3PTWT&pd_rd_wg=815Vo&pd_rd_r=860fb88a-be53-4daf-8d0d-39904228e97a&th=1&psc=1",
"https://www.amazon.in/QUTUN-Premium-Waffle-Henley-Tshirt/dp/B0H74XDHMM/ref=sr_1_8?crid=1ENB8I08MQZHW&dib=eyJ2IjoiMSJ9.pNDhyp9aWJr_Brl-RhDjb6GsMWYHe4sVJGGX7D7BTfeJZOyR6_vqnYYEbCky3zqq5dn_sL9DqvtbLXwxeBn27d3RZdJCiJdP4vL6bmNyAe_2r3XIJtNV6nNZXGlD5ciD6Om7n-DrYOM6UhoXNZkJ0ghzBGIb0z6xcgjvueaHEu7syNOfdHJULhqenp4YDzMvHVTtiAUXSSAmUP98Hx130Ou14tnR_k1FsDKD0ntgh-2w9OfGLXrubO2VNCCfpR7DOfmbLeT2GLNMdzYDO0aC1zmiSKlZYGBu0oaXVC62dFA.R3qnVZYfRCU9OXfWyH0UMKI4n5gfFrTBCrcovG6eYBI&dib_tag=se&keywords=tshirt%2Bshirts%2Bmen&qid=1789134778&sprefix=tshi%2Caps%2C473&sr=8-8&th=1&psc=1",
"https://www.amazon.in/GAP-Graphic-Printed-Sleeves-T-Shirt/dp/B0GK1BDZR7/ref=sr_1_1_sspa?crid=1ENB8I08MQZHW&dib=eyJ2IjoiMSJ9.pNDhyp9aWJr_Brl-RhDjb6GsMWYHe4sVJGGX7D7BTfeJZOyR6_vqnYYEbCky3zqq5dn_sL9DqvtbLXwxeBn27d3RZdJCiJdP4vL6bmNyAe_2r3XIJtNV6nNZXGlD5ciD6Om7n-DrYOM6UhoXNZkJ0ghzBGIb0z6xcgjvueaHEu7syNOfdHJULhqenp4YDzMvHVTtiAUXSSAmUP98Hx130Ou14tnR_k1FsDKD0ntgh-2w9OfGLXrubO2VNCCfpR7DOfmbLeT2GLNMdzYDO0aC1zmiSKlZYGBu0oaXVC62dFA.R3qnVZYfRCU9OXfWyH0UMKI4n5gfFrTBCrcovG6eYBI&dib_tag=se&keywords=tshirt%2Bshirts%2Bmen&qid=1789134778&sprefix=tshi%2Caps%2C473&sr=8-1-spons&aref=PJmTmYdMC3&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGY&th=1&psc=1",
"https://www.amazon.in/AUSK-Textured-T-Shirt-Regular-Stylish/dp/B0H9LJZF4R/ref=sr_1_10?crid=1ENB8I08MQZHW&dib=eyJ2IjoiMSJ9.pNDhyp9aWJr_Brl-RhDjb6GsMWYHe4sVJGGX7D7BTfeJZOyR6_vqnYYEbCky3zqq5dn_sL9DqvtbLXwxeBn27d3RZdJCiJdP4vL6bmNyAe_2r3XIJtNV6nNZXGlD5ciD6Om7n-DrYOM6UhoXNZkJ0ghzBGIb0z6xcgjvueaHEu7syNOfdHJULhqenp4YDzMvHVTtiAUXSSAmUP98Hx130Ou14tnR_k1FsDKD0ntgh-2w9OfGLXrubO2VNCCfpR7DOfmbLeT2GLNMdzYDO0aC1zmiSKlZYGBu0oaXVC62dFA.R3qnVZYfRCU9OXfWyH0UMKI4n5gfFrTBCrcovG6eYBI&dib_tag=se&keywords=tshirt%2Bshirts%2Bmen&qid=1789134778&sprefix=tshi%2Caps%2C473&sr=8-10&th=1&psc=1"
]

In [22]:
products = []

for url in PRODUCT_URLS:
    product = scrape_product(url)
    
    if product is not None:
        products.append(product)

products

[{'title': 'TVS Originals 100% Cotton Overdyed Polo T-Shirt for Men | Heavyweight 240 GSM | Washed Finish for Vintage Look | Classic Fit with Side Slits | Soft Ribbed Collar | Durable, Pre-Washed, Premium Feel',
  'price': 1094.0,
  'rating': 1.0,
  'reviews': None,
  'availability': 'In stock',
  'url': 'https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd_plhdr=t&hsa_cr_id=0&qid=1789134778&sr=1-1-941ebe59-d25b-49d0-b214-12cc5b66c90f&i=aps&aref=wsdpJlJGBJ&_encoding=UTF8&ref_=sbx_s_sparkle_sbtcd_asin_0_title&pd_rd_w=hkVwn&content-id=amzn1.sym.7b6188a8-103f-4f88-b43a-19aba06e30c1%3Aamzn1.sym.7b6188a8-103f-4f88-b43a-19aba06e30c1&pf_rd_p=7b6188a8-103f-4f88-b43a-19aba06e30c1&pf_rd_r=NSM1EVTP5VX9ACY3PTWT&pd_rd_wg=815Vo&pd_rd_r=860fb88a-be53-4daf-8d0d-39904228e97a&th=1&psc=1',
  'scraped_at': datetime.datetime(2026, 9, 11, 19, 29, 11, 520040)},
 {'title': "Men's Premium Waffle Knit Henley Tshirt for Men",
  'price': 349.0,
  'rating': 4.2,
  'reviews': None,
  'availability': 'In stock',
  'url': 

In [23]:
#To see more properly
df = pd.DataFrame(products)

df

,title,price,rating,reviews,availability,url,scraped_at
0,TVS Originals 100% Cotton Overdyed Polo T-Shir...,1094.0,1.0,None,In stock,https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd...,2026-09-11 19:29:11.520040
1,Men's Premium Waffle Knit Henley Tshirt for Men,349.0,4.2,None,In stock,https://www.amazon.in/QUTUN-Premium-Waffle-Hen...,2026-09-11 19:29:14.833435
2,Gap Men Graphic Printed Slim Fit Crew Neck Sho...,569.0,4.0,None,In stock,https://www.amazon.in/GAP-Graphic-Printed-Slee...,2026-09-11 19:29:18.296306
3,AUSK Men's Cotton Blend Rib Knit Textured Full...,399.0,3.2,None,In stock,https://www.amazon.in/AUSK-Textured-T-Shirt-Re...,2026-09-11 19:29:21.658339


In [24]:
#Creating price history 

from datetime import datetime
import os

price_history = []

for product in products:
    if product is not None:
        price_history.append({
            "title": product["title"],
            "price": product["price"],
            "scraped_at": product["scraped_at"],
            "url": product["url"]
        })

price_history_df = pd.DataFrame(price_history)

# Save price history
price_history_df.to_csv(
    "price_history.csv",
    mode="a",
    header=not os.path.exists("price_history.csv"),
    index=False
)

print("Price history saved successfully!")
print(price_history_df)

Price history saved successfully!
                                               title   price  \
0  TVS Originals 100% Cotton Overdyed Polo T-Shir...  1094.0   
1    Men's Premium Waffle Knit Henley Tshirt for Men   349.0   
2  Gap Men Graphic Printed Slim Fit Crew Neck Sho...   569.0   
3  AUSK Men's Cotton Blend Rib Knit Textured Full...   399.0   

                  scraped_at  \
0 2026-09-11 19:29:11.520040   
1 2026-09-11 19:29:14.833435   
2 2026-09-11 19:29:18.296306   
3 2026-09-11 19:29:21.658339   

                                                 url  
0  https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd...  
1  https://www.amazon.in/QUTUN-Premium-Waffle-Hen...  
2  https://www.amazon.in/GAP-Graphic-Printed-Slee...  
3  https://www.amazon.in/AUSK-Textured-T-Shirt-Re...  


In [25]:
#saving products

df.to_csv(
    "all_products.csv",
    index=False
)

print("All products saved successfully!")
print("File: all_products.csv")

All products saved successfully!
File: all_products.csv


In [26]:
#Loading the historical data

price_history_df = pd.read_csv("price_history.csv")

print("Historical data loaded successfully!")
print(price_history_df)

Historical data loaded successfully!
                                               title   price  \
0  TVS Originals 100% Cotton Overdyed Polo T-Shir...  1094.0   
1    Men's Premium Waffle Knit Henley Tshirt for Men   349.0   
2  Gap Men Graphic Printed Slim Fit Crew Neck Sho...   569.0   
3  AUSK Men's Cotton Blend Rib Knit Textured Full...   399.0   

                   scraped_at  \
0  2026-09-11 19:29:11.520040   
1  2026-09-11 19:29:14.833435   
2  2026-09-11 19:29:18.296306   
3  2026-09-11 19:29:21.658339   

                                                 url  
0  https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd...  
1  https://www.amazon.in/QUTUN-Premium-Waffle-Hen...  
2  https://www.amazon.in/GAP-Graphic-Printed-Slee...  
3  https://www.amazon.in/AUSK-Textured-T-Shirt-Re...  


In [27]:
#to check the data structure
price_history_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   title       4 non-null      str    
 1   price       4 non-null      float64
 2   scraped_at  4 non-null      str    
 3   url         4 non-null      str    
dtypes: float64(1), str(3)
memory usage: 3.0 KB


In [28]:
#to check latest record
price_history_df.tail()

,title,price,scraped_at,url
0,TVS Originals 100% Cotton Overdyed Polo T-Shir...,1094.0,2026-09-11 19:29:11.520040,https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd...
1,Men's Premium Waffle Knit Henley Tshirt for Men,349.0,2026-09-11 19:29:14.833435,https://www.amazon.in/QUTUN-Premium-Waffle-Hen...
2,Gap Men Graphic Printed Slim Fit Crew Neck Sho...,569.0,2026-09-11 19:29:18.296306,https://www.amazon.in/GAP-Graphic-Printed-Slee...
3,AUSK Men's Cotton Blend Rib Knit Textured Full...,399.0,2026-09-11 19:29:21.658339,https://www.amazon.in/AUSK-Textured-T-Shirt-Re...


AttributeError: 'list' object has no attribute 'groupby'

In [30]:
#minimum price

lowest_prices = (
    price_history_df
    .groupby(["url", "title"])["price"]
    .min()
    .reset_index()
)

lowest_prices

,url,title,price
0,https://www.amazon.in/AUSK-Textured-T-Shirt-Re...,AUSK Men's Cotton Blend Rib Knit Textured Full...,399.0
1,https://www.amazon.in/GAP-Graphic-Printed-Slee...,Gap Men Graphic Printed Slim Fit Crew Neck Sho...,569.0
2,https://www.amazon.in/QUTUN-Premium-Waffle-Hen...,Men's Premium Waffle Knit Henley Tshirt for Men,349.0
3,https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd...,TVS Originals 100% Cotton Overdyed Polo T-Shir...,1094.0


In [31]:
lowest_prices = lowest_prices.rename(
    columns={"price": "minimum_historical_price"}
)

lowest_prices

,url,title,minimum_historical_price
0,https://www.amazon.in/AUSK-Textured-T-Shirt-Re...,AUSK Men's Cotton Blend Rib Knit Textured Full...,399.0
1,https://www.amazon.in/GAP-Graphic-Printed-Slee...,Gap Men Graphic Printed Slim Fit Crew Neck Sho...,569.0
2,https://www.amazon.in/QUTUN-Premium-Waffle-Hen...,Men's Premium Waffle Knit Henley Tshirt for Men,349.0
3,https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd...,TVS Originals 100% Cotton Overdyed Polo T-Shir...,1094.0


In [32]:
# maximum and average historical price

price_stats = (
    price_history_df
    .groupby(["url", "title"])["price"]
    .agg(["min", "max", "mean"])
    .reset_index()
)

price_stats = price_stats.rename(columns={
    "min": "minimum_price",
    "max": "maximum_price",
    "mean": "average_price"
})

price_stats

,url,title,minimum_price,maximum_price,average_price
0,https://www.amazon.in/AUSK-Textured-T-Shirt-Re...,AUSK Men's Cotton Blend Rib Knit Textured Full...,399.0,399.0,399.0
1,https://www.amazon.in/GAP-Graphic-Printed-Slee...,Gap Men Graphic Printed Slim Fit Crew Neck Sho...,569.0,569.0,569.0
2,https://www.amazon.in/QUTUN-Premium-Waffle-Hen...,Men's Premium Waffle Knit Henley Tshirt for Men,349.0,349.0,349.0
3,https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd...,TVS Originals 100% Cotton Overdyed Polo T-Shir...,1094.0,1094.0,1094.0


In [33]:
# average price and price change

price_analysis = (
    price_history_df
    .groupby(["url", "title"])["price"]
    .agg(["mean"])
    .reset_index()
)

price_analysis = price_analysis.rename(
    columns={"mean": "average_price"}
)

# Merge current prices
price_analysis = price_analysis.merge(
    df[["url", "price"]],
    on="url",
    how="left"
)

price_analysis = price_analysis.rename(
    columns={"price": "current_price"}
)

# Calculate price change
price_analysis["price_change"] = (
    price_analysis["current_price"]
    - price_analysis["average_price"]
)

# Round values
price_analysis["average_price"] = price_analysis["average_price"].round(2)
price_analysis["price_change"] = price_analysis["price_change"].round(2)

price_analysis

,url,title,average_price,current_price,price_change
0,https://www.amazon.in/AUSK-Textured-T-Shirt-Re...,AUSK Men's Cotton Blend Rib Knit Textured Full...,399.0,399.0,0.0
1,https://www.amazon.in/GAP-Graphic-Printed-Slee...,Gap Men Graphic Printed Slim Fit Crew Neck Sho...,569.0,569.0,0.0
2,https://www.amazon.in/QUTUN-Premium-Waffle-Hen...,Men's Premium Waffle Knit Henley Tshirt for Men,349.0,349.0,0.0
3,https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd...,TVS Originals 100% Cotton Overdyed Polo T-Shir...,1094.0,1094.0,0.0


In [38]:
# Load the actual historical CSV

price_history_df = pd.read_csv("price_history.csv")

# Make sure the data is sorted
price_history_df = price_history_df.sort_values(
    ["url", "scraped_at"]
)

# Get previous price for each product
price_history_df["previous_price"] = (
    price_history_df
    .groupby("url")["price"]
    .shift(1)
)

# Calculate price change
price_history_df["price_change"] = (
    price_history_df["price"]
    - price_history_df["previous_price"]
)

# Calculate percentage change
price_history_df["price_change_percent"] = (
    price_history_df["price_change"]
    / price_history_df["previous_price"]
) * 100

price_history_df[
    [
        "title",
        "price",
        "previous_price",
        "price_change",
        "price_change_percent",
        "scraped_at"
    ]
]

,title,price,previous_price,price_change,price_change_percent,scraped_at
3,AUSK Men's Cotton Blend Rib Knit Textured Full...,399.0,NaN,NaN,NaN,2026-09-11 19:29:21.658339
2,Gap Men Graphic Printed Slim Fit Crew Neck Sho...,569.0,NaN,NaN,NaN,2026-09-11 19:29:18.296306
1,Men's Premium Waffle Knit Henley Tshirt for Men,349.0,NaN,NaN,NaN,2026-09-11 19:29:14.833435
0,TVS Originals 100% Cotton Overdyed Polo T-Shir...,1094.0,NaN,NaN,NaN,2026-09-11 19:29:11.520040


In [39]:
biggest_drops = (
    price_history_df[
        price_history_df["price_change"] < 0
    ]
    .sort_values("price_change")
    .head(10)
)

biggest_drops[
    [
        "title",
        "price",
        "previous_price",
        "price_change",
        "price_change_percent",
        "scraped_at"
    ]
]

,title,price,previous_price,price_change,price_change_percent,scraped_at


In [40]:
# Add target prices for each product

target_prices = {
    "https://www.amazon.in/dp/B0FH9J9VXY": 1000,
    "https://www.amazon.in/dp/B0H74XDHMM": 1500,
    "https://www.amazon.in/dp/B0GK1BDZR7": 2000,
    "https://www.amazon.in/dp/B0H9LJZF4R": 1200
}

df["target_price"] = df["url"].map(target_prices)

df

,title,price,rating,reviews,availability,url,scraped_at,target_price
0,TVS Originals 100% Cotton Overdyed Polo T-Shir...,1094.0,1.0,None,In stock,https://www.amazon.in/gp/aw/d/B0FH9J9VXY?pd_rd...,2026-09-11 19:29:11.520040,NaN
1,Men's Premium Waffle Knit Henley Tshirt for Men,349.0,4.2,None,In stock,https://www.amazon.in/QUTUN-Premium-Waffle-Hen...,2026-09-11 19:29:14.833435,NaN
2,Gap Men Graphic Printed Slim Fit Crew Neck Sho...,569.0,4.0,None,In stock,https://www.amazon.in/GAP-Graphic-Printed-Slee...,2026-09-11 19:29:18.296306,NaN
3,AUSK Men's Cotton Blend Rib Knit Textured Full...,399.0,3.2,None,In stock,https://www.amazon.in/AUSK-Textured-T-Shirt-Re...,2026-09-11 19:29:21.658339,NaN


In [41]:
df.to_csv("all_products.csv", index=False)

print("Target prices added and saved successfully!")

Target prices added and saved successfully!


In [42]:
# Price Alert Checker

alerts = []

for _, product in df.iterrows():

    if pd.notna(product["price"]) and pd.notna(product["target_price"]):

        if product["price"] <= product["target_price"]:
            alerts.append({
                "title": product["title"],
                "current_price": product["price"],
                "target_price": product["target_price"],
                "url": product["url"]
            })

price_alerts = pd.DataFrame(alerts)

if not price_alerts.empty:
    print("🚨 PRICE ALERT!")
    print(price_alerts)
else:
    print("No products have reached their target price yet.")

No products have reached their target price yet.


In [43]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

sender_email = "vedantbedmutha@gmail.com"
receiver_email = "vedantbedmutha@gmail.com"
app_password = "YOUR_GMAIL_APP_PASSWORD"


def send_price_alert(product):

    subject = "🚨 Amazon Price Alert!"

    body = f"""
Price Alert!

Product:
{product['title']}

Current Price: ₹{product['current_price']}
Target Price: ₹{product['target_price']}

Buy it here:
{product['url']}
"""

    message = MIMEMultipart()
    message["From"] = sender_email
    message["To"] = receiver_email
    message["Subject"] = subject

    message.attach(MIMEText(body, "plain"))

    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(sender_email, app_password)
            server.send_message(message)

        print("✅ Email sent successfully!")

    except Exception as e:
        print("❌ Email failed:", e)


if not price_alerts.empty:

    for _, product in price_alerts.iterrows():
        send_price_alert(product)

else:
    print("No products have reached their target price.")

No products have reached their target price.


In [44]:
df.to_csv("all_products.csv", index=False)

print("all_products.csv created successfully!")

all_products.csv created successfully!
